# M4 — Escalation risk  (Anna's method: time split + tuned threshold + PR-AUC)

Based on **Anna**'s escalation model. Class-balanced logistic regression on
summary + tags + category + channel + month + tenure + tag-count. A
**chronological 70/15/15 split**; the decision **threshold is tuned on the
validation set** (maximising F1 of the rare escalation class) and PR-AUC /
ROC-AUC are reported on the test set. `sla_breach` and outcome fields are
excluded. Saves `models/m4_escalation.joblib`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import northstar_prep as P
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                             precision_score, recall_score)

df = P.load_clean()
df["escalated"] = df["escalated"].astype(bool)
FEATS, TARGET = ["case_summary","tags","category","channel","month_num",
                 "customer_tenure_months","n_tags"], "escalated"
tr, va, te = P.time_split(df)
trans = [P.t_word(), P.t_char(), P.t_tags(), P.t_cat(["category","channel"]),
         P.t_num(["month_num","customer_tenure_months","n_tags"])]
pipe = P.make_clf(trans).fit(tr[FEATS], tr[TARGET])
pos = list(pipe.classes_).index(True)

In [ ]:
# tune threshold on validation to maximise F1 of the escalation (True) class
pv = pipe.predict_proba(va[FEATS])[:, pos]
grid = np.linspace(0.1, 0.9, 81)
thr = float(max(grid, key=lambda t: f1_score(va[TARGET], pv >= t, zero_division=0)))

pt = pipe.predict_proba(te[FEATS])[:, pos]
yhat = pt >= thr
metrics = {"threshold": round(thr, 3),
           "ROC_AUC": round(roc_auc_score(te[TARGET], pt), 3),
           "PR_AUC":  round(average_precision_score(te[TARGET], pt), 3),
           "recall_escalate":    round(recall_score(te[TARGET], yhat, zero_division=0), 3),
           "precision_escalate": round(precision_score(te[TARGET], yhat, zero_division=0), 3),
           "macro_F1": round(f1_score(te[TARGET], yhat, average="macro"), 3),
           "baseline_acc": round((~df[TARGET]).mean(), 3)}
print(metrics)

In [ ]:
test = te[["case_id","case_summary",TARGET]].copy().reset_index(drop=True)
test["prob"]      = pt.round(3)
test["predicted"] = yhat
test["confidence"] = test["prob"]

full = P.make_clf(trans).fit(df[FEATS], df[TARGET])     # deploy model (threshold reused)
path = P.save_bundle("m4_escalation", {
    "model": full, "target": TARGET, "features": FEATS, "classes": [False, True],
    "kind": "binary", "label": "Escalation risk", "threshold": round(thr, 3),
    "metrics": metrics, "test": test,
    "meta": {"author":"Anna","split":"time 70/15/15","threshold_tuned_on":"validation",
             "inputs":"summary + tags + category + channel + month + tenure + n_tags"}})
print("saved", path, "| threshold", round(thr,3), "| test rows:", len(test))